# Ball Detection & Tracking — Data Preparation Pipeline

Import Libraries

In [ ]:
import cv2
import numpy as np
import pandas as pd
import torch
from google.colab import drive
import os

Mount Google Drive

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading & Extracting the Data

In [ ]:
!find /content/drive/MyDrive -iname "*TrackNetV2*"
!unzip -q "/content/drive/MyDrive/TrackNetV2.zip" -d /content/TrackNetV2

/content/drive/MyDrive/TrackNetV2.zip
replace /content/TrackNetV2/TrackNetV2/Amateur/match1/csv/1_00_01_ball.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [ ]:
!find /content/TrackNetV2 -maxdepth 4 | head -30

/content/TrackNetV2
/content/TrackNetV2/TrackNetV2
/content/TrackNetV2/TrackNetV2/Professional
/content/TrackNetV2/TrackNetV2/Professional/match16
/content/TrackNetV2/TrackNetV2/Professional/match16/video
/content/TrackNetV2/TrackNetV2/Professional/match16/csv
/content/TrackNetV2/TrackNetV2/Professional/match17
/content/TrackNetV2/TrackNetV2/Professional/match17/video
/content/TrackNetV2/TrackNetV2/Professional/match17/csv
/content/TrackNetV2/TrackNetV2/Professional/match15
/content/TrackNetV2/TrackNetV2/Professional/match15/video
/content/TrackNetV2/TrackNetV2/Professional/match15/csv
/content/TrackNetV2/TrackNetV2/Professional/match12
/content/TrackNetV2/TrackNetV2/Professional/match12/video
/content/TrackNetV2/TrackNetV2/Professional/match12/csv
/content/TrackNetV2/TrackNetV2/Professional/match20
/content/TrackNetV2/TrackNetV2/Professional/match20/video
/content/TrackNetV2/TrackNetV2/Professional/match20/csv
/content/TrackNetV2/TrackNetV2/Professional/match14
/content/TrackNetV2/Tra

Inspect Labels

In [ ]:
LABEL_DIR = "/content/TrackNetV2/TrackNetV2/Amateur/match1/csv/1_00_01_ball.csv"
label = pd.read_csv(LABEL_DIR)
label.head(20)

,Frame,Visibility,X,Y
0,0,0,0,0
1,1,0,0,0
2,2,0,0,0
3,3,0,0,0
4,4,0,0,0
5,5,0,0,0
6,6,0,0,0
7,7,0,0,0
8,8,0,0,0
9,9,0,0,0


In [ ]:
print(label['Visibility'].unique())
print(label.columns.tolist())

[0 1]
['Frame', 'Visibility', 'X', 'Y']


Inspect Video Properties

In [ ]:
IMG_PATH = "/content/TrackNetV2/TrackNetV2/Amateur/match1/video/1_00_01.mp4"
video = cv2.VideoCapture(IMG_PATH)
if not video.isOpened():
    print("Error: Could not open video.")
else:
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    fps    = video.get(cv2.CAP_PROP_FPS)
    print(f"Resolution : {width} X {height}")
    print(f"FPS:         {fps}")

Resolution : 1280 X 720
FPS:         30.0


In [ ]:
!ls /content/TrackNetV2/TrackNetV2/Amateur/match1/video
!ls /content/TrackNetV2/TrackNetV2/Amateur/match1/csv

1_00_01.mp4  1_01_02.mp4  1_01_04.mp4  1_03_04.mp4  1_04_05.mp4
1_01_01.mp4  1_01_03.mp4  1_02_04.mp4  1_03_05.mp4  1_05_05.mp4
1_00_01_ball.csv  1_01_03_ball.csv  1_03_04_ball.csv  1_05_05_ball.csv
1_01_01_ball.csv  1_01_04_ball.csv  1_03_05_ball.csv
1_01_02_ball.csv  1_02_04_ball.csv  1_04_05_ball.csv


Extract Frames from Videos

In [ ]:
VIDEOS_DIR = "/content/TrackNetV2/TrackNetV2/"
FRAMES_BASE_OUTPUT = "/content/frames_output"

def extract_frames(video_path, output_dir):
  os.makedirs(output_dir, exist_ok=True)
  cap = cv2.VideoCapture(video_path)
  if not cap.isOpened():
      print(f"Error: Could not open video {video_path}")
      return 0

  print(f"Extracting frames from: {video_path} to {output_dir}")
  i = 0
  while True:
    ret, frame = cap.read()
    if not ret:
      break
    cv2.imwrite(os.path.join(output_dir, f"frame_{i:05d}.jpg"), frame)
    i += 1
  cap.release()
  print(f"Extracted {i} frames.")
  return i

for dirpath, dirnames, filenames in os.walk(VIDEOS_DIR):
  for filename in filenames:
    if filename.endswith(".mp4"):
      video_file_path = os.path.join(dirpath, filename)
      relative_path = os.path.relpath(video_file_path, VIDEOS_DIR)
      video_output_dir = os.path.join(FRAMES_BASE_OUTPUT, os.path.dirname(relative_path), os.path.splitext(filename)[0])

      extract_frames(video_file_path, video_output_dir)


Error: Could not open video /content/TrackNetV2/TrackNetV2/Professional/match16/video/2_08_08.mp4
Extracting frames from: /content/TrackNetV2/TrackNetV2/Professional/match16/video/3_14_09.mp4 to /content/frames_output/Professional/match16/video/3_14_09
Extracted 342 frames.
Extracting frames from: /content/TrackNetV2/TrackNetV2/Professional/match16/video/1_13_20.mp4 to /content/frames_output/Professional/match16/video/1_13_20
Extracted 438 frames.
Extracting frames from: /content/TrackNetV2/TrackNetV2/Professional/match16/video/3_12_06.mp4 to /content/frames_output/Professional/match16/video/3_12_06
Extracted 655 frames.
Extracting frames from: /content/TrackNetV2/TrackNetV2/Professional/match16/video/3_17_16.mp4 to /content/frames_output/Professional/match16/video/3_17_16
Extracted 435 frames.
Extracting frames from: /content/TrackNetV2/TrackNetV2/Professional/match16/video/1_03_06.mp4 to /content/frames_output/Professional/match16/video/1_03_06
Extracted 596 frames.
Extracting frames

Build Sliding-Window Frame Triplets (Truncated to CSV Length)

In [ ]:
WINDOW_SIZE = 3

samples = []

for dirpath, dirnames, filenames in os.walk(FRAMES_BASE_OUTPUT):
    if dirnames:
        continue

    frame_files = sorted(f for f in filenames if f.lower().endswith((".jpg", ".jpeg", ".png")))
    if not frame_files:
        continue

    rel_path = os.path.relpath(dirpath, FRAMES_BASE_OUTPUT)
    parts = rel_path.split(os.sep)
    clip_name = parts[-1]
    csv_parts = ["csv" if p == "video" else p for p in parts[:-1]]
    csv_path = os.path.join(VIDEOS_DIR, *csv_parts, f"{clip_name}_ball.csv")

    if not os.path.exists(csv_path):
        continue

    clip_labels = pd.read_csv(csv_path)

    n = len(clip_labels)
    frame_paths = [os.path.join(dirpath, f) for f in frame_files[:n]]

    for i in range(len(frame_paths) - WINDOW_SIZE + 1):
        frame_path_1, frame_path_2, frame_path_3 = frame_paths[i], frame_paths[i + 1], frame_paths[i + 2]
        label_1 = tuple(clip_labels.iloc[i][["Visibility", "X", "Y"]])
        label_2 = tuple(clip_labels.iloc[i + 1][["Visibility", "X", "Y"]])
        label_3 = tuple(clip_labels.iloc[i + 2][["Visibility", "X", "Y"]])
        samples.append((frame_path_1, frame_path_2, frame_path_3, label_1, label_2, label_3))

print(f"Built {len(samples)} sliding-window triplets from {FRAMES_BASE_OUTPUT}")

Built 47949 sliding-window triplets from /content/frames_output


In [ ]:
class TrackNetDataset(torch.utils.data.Dataset):
    def __init__(self, samples, resize_to=(512, 288)):
      self.samples = samples
      self.resize_to = resize_to  # (new_w, new_h)

    def __len__(self):
      return len(self.samples)

    def __getitem__(self, idx):
      frame_path_1, frame_path_2, frame_path_3, label_1, label_2, label_3 = self.samples[idx]

      new_w, new_h = self.resize_to
      images = []
      labels = []
      for frame_path, label in zip((frame_path_1, frame_path_2, frame_path_3), (label_1, label_2, label_3)):
        image = cv2.imread(frame_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = image.shape[:2]
        image = cv2.resize(image, self.resize_to, interpolation=cv2.INTER_AREA)
        images.append(image.transpose(2, 0, 1))  # (H, W, C) -> (C, H, W)

        visibility, x, y = label
        scale_x = new_w / orig_w
        scale_y = new_h / orig_h
        labels.append((visibility, x * scale_x, y * scale_y))

      images = np.concatenate(images, axis=0)  # 3 x (3, H, W) -> (9, H, W)
      labels = np.array(labels, dtype=np.float32)

      return torch.from_numpy(images), torch.from_numpy(labels)
